# AxelSMOTE Master Research Notebook

One self-contained Google Colab notebook for the AxelSMOTE project.

This notebook preserves the current custom AxelSMOTE implementation, the binary real-dataset benchmark, controlled experiments, persistence to Google Drive, summaries, plots, and final verification.

**Important research note:** the AxelSMOTE implementation below is the project's current custom implementation based on the current project interpretation. It is **not claimed to be the authors' official implementation**. Exact paper-level fidelity should be checked against the original paper, especially trait construction, similarity, mutation/noise, and parameter semantics.

Core rules:
- Test data is never oversampled.
- Scaling and sampling are fitted only on training data inside an imblearn pipeline.
- The real benchmark uses all 10 seeds.
- Results are saved to Google Drive.
- Existing complete result files are reused; incomplete/corrupt files are rerun.
- Results are reported objectively, including unfavorable outcomes.

## 1. Project Overview

Primary binary benchmark:

**4 datasets × 3 methods × 10 seeds = 120 rows**

Datasets:
- Wisconsin
- KC1
- ADS
- ILPD

Methods:
- Class Weight
- SMOTE
- AxelSMOTE

Additional controlled experiments:
- Basic benchmark
- Imbalance ratio
- Dimensionality
- Class overlap
- AxelSMOTE one-at-a-time sensitivity
- Classifier extension with Logistic Regression, Random Forest, and SVM

Thyroid is handled defensively because its current OpenML lookup fails. Glass, Page Blocks, and Ecoli are kept outside the current binary Logistic Regression benchmark because they are multiclass.

## 2. Installation

In [ ]:
!pip -q install --upgrade --force-reinstall \
    "numpy==2.2.6" \
    "scipy==1.15.3" \
    "pandas==2.2.3" \
    "scikit-learn==1.6.1" \
    "imbalanced-learn==0.13.0" \
    "matplotlib==3.10.1" \
    "seaborn==0.13.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.1/130.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.4/238.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 384.7/384.7 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.1/306.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/

## 3. Imports

In [ ]:
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from sklearn.base import BaseEstimator
from sklearn.datasets import fetch_openml, make_classification
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
)

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")

print("Imports complete.")

/usr/local/lib/python3.13/dist-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.3)
  from scipy.sparse import csr_matrix, issparse


Imports complete.


## 4. Configuration

In [ ]:
OPENML_NAMES = {
    "wisconsin": "breast-w",
    "thyroid": "thyroid",
    "kc1": "kc1",
    "ads": "Internet-Advertisements",
    "ilpd": "ilpd",
    "glass": "glass",
    "page_blocks": "page-blocks",
    "ecoli": "ecoli",
}

# Current binary benchmark only.
REAL_DATASETS = ["wisconsin", "kc1", "ads", "ilpd"]

METHODS = ["class_weight", "smote", "axelsmote"]
SEEDS = list(range(10))

AXEL_PARAMS = {
    "k_neighbors": 2,
    "n_traits": 4,
    "theta": 0.4,
    "alpha": 0.4,
}

IMBALANCE_RATIOS = [2, 5, 10, 20]
DIMENSIONS = [5, 10, 20, 50, 100]
CLASS_SEPARATIONS = [0.3, 0.6, 1.0, 1.5, 2.0]

SENSITIVITY_VALUES = {
    "k_neighbors": [1, 2, 3, 5],
    "n_traits": [2, 4, 6, 8],
    "theta": [0.2, 0.4, 0.6, 0.8],
    "alpha": [0.2, 0.4, 0.6, 0.8],
}

# Fixed seeds for controlled experiments too.
CONTROLLED_SEEDS = list(range(10))

DRIVE_ROOT = Path("/content/drive/MyDrive/AxelSMOTE")
RESULTS_DIR = DRIVE_ROOT / "results"
FIGURES_DIR = DRIVE_ROOT / "figures"
TABLES_DIR = DRIVE_ROOT / "final_tables"

for p in [RESULTS_DIR, FIGURES_DIR, TABLES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")

Configuration loaded.


## 5. Google Drive Setup

In [ ]:
!rm -rf /content/drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

for p in [RESULTS_DIR, FIGURES_DIR, TABLES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Drive mounted.")
print("Project root:", DRIVE_ROOT)

Mounted at /content/drive
Drive mounted.
Project root: /content/drive/MyDrive/AxelSMOTE


### Persistence helpers

In [ ]:
def save_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"Saved {len(df):,} rows -> {path}")


def load_complete_csv(path, expected_rows=None, required_columns=None):
    path = Path(path)

    if not path.exists() or path.stat().st_size == 0:
        return None

    try:
        df = pd.read_csv(path)

        if expected_rows is not None and len(df) != expected_rows:
            print(
                f"Existing file is incomplete: {path} "
                f"({len(df)} rows; expected {expected_rows})."
            )
            return None

        if required_columns is not None:
            missing = set(required_columns) - set(df.columns)
            if missing:
                print(f"Existing file is missing columns: {sorted(missing)}")
                return None

        if df.empty:
            return None

        print(f"Reusing complete result: {path}")
        return df

    except Exception as exc:
        print(f"Could not reuse {path}: {exc}")
        return None


def save_checkpoint_rows(rows, path):
    df = pd.DataFrame(rows)
    save_csv(df, path)
    return df


def expected_result_file(name):
    return RESULTS_DIR / name

## 6. Dataset Loading

In [ ]:
def load_dataset(key):
    # Existing project loader, retained with defensive key checking.
    if key not in OPENML_NAMES:
        raise KeyError(f"Unknown dataset key: {key}")

    ds = fetch_openml(
        name=OPENML_NAMES[key],
        version="active",
        as_frame=True
    )

    X = pd.get_dummies(ds.data, drop_first=False)
    X = X.apply(pd.to_numeric, errors="coerce").fillna(0)

    y = pd.Series(ds.target).astype("category").cat.codes.to_numpy()

    return X.to_numpy(float), y


def class_distribution(y):
    return pd.Series(y).value_counts().sort_index()


def summarize_dataset(key, X, y):
    return {
        "dataset": key,
        "n_samples": X.shape[0],
        "n_features": X.shape[1],
        "n_classes": len(np.unique(y)),
        "class_distribution": str(class_distribution(y).to_dict()),
    }

## 7. Dataset Validation

In [ ]:
VALIDATION_ROWS = []

for key in OPENML_NAMES:
    try:
        X, y = load_dataset(key)
        row = summarize_dataset(key, X, y)
        row["status"] = "OK"
        VALIDATION_ROWS.append(row)

        print(
            f"{key:12s} X={X.shape}, "
            f"classes={class_distribution(y).to_dict()}"
        )

    except Exception as exc:
        VALIDATION_ROWS.append({
            "dataset": key,
            "n_samples": np.nan,
            "n_features": np.nan,
            "n_classes": np.nan,
            "class_distribution": "",
            "status": f"FAILED: {type(exc).__name__}: {exc}",
        })
        print(f"{key:12s} FAILED: {type(exc).__name__}: {exc}")

validation_df = pd.DataFrame(VALIDATION_ROWS)
display(validation_df)

wisconsin    X=(699, 9), classes={0: 458, 1: 241}
thyroid      FAILED: OpenMLError: No active dataset thyroid found.
kc1          X=(2109, 21), classes={0: 1783, 1: 326}
ads          X=(3279, 3113), classes={0: 459, 1: 2820}
ilpd         X=(583, 11), classes={0: 416, 1: 167}
glass        X=(214, 9), classes={0: 70, 1: 76, 2: 17, 3: 13, 4: 29, 5: 9}
page_blocks  X=(5473, 10), classes={0: 4913, 1: 329, 2: 28, 3: 88, 4: 115}
ecoli        X=(336, 7), classes={0: 143, 1: 77, 2: 2, 3: 2, 4: 35, 5: 20, 6: 5, 7: 52}


,dataset,n_samples,n_features,n_classes,class_distribution,status
0,wisconsin,699.0,9.0,2.0,"{0: 458, 1: 241}",OK
1,thyroid,NaN,NaN,NaN,,FAILED: OpenMLError: No active dataset thyroid...
2,kc1,2109.0,21.0,2.0,"{0: 1783, 1: 326}",OK
3,ads,3279.0,3113.0,2.0,"{0: 459, 1: 2820}",OK
4,ilpd,583.0,11.0,2.0,"{0: 416, 1: 167}",OK
5,glass,214.0,9.0,6.0,"{0: 70, 1: 76, 2: 17, 3: 13, 4: 29, 5: 9}",OK
6,page_blocks,5473.0,10.0,5.0,"{0: 4913, 1: 329, 2: 28, 3: 88, 4: 115}",OK
7,ecoli,336.0,7.0,8.0,"{0: 143, 1: 77, 2: 2, 3: 2, 4: 35, 5: 20, 6: 5...",OK


### Binary benchmark eligibility

In [ ]:
for key in REAL_DATASETS:
    row = validation_df.loc[
        validation_df["dataset"] == key
    ].iloc[0]

    if not str(row["status"]).startswith("OK"):
        raise RuntimeError(
            f"Required dataset '{key}' could not be loaded: {row['status']}"
        )

    X_tmp, y_tmp = load_dataset(key)

    if len(np.unique(y_tmp)) != 2:
        raise RuntimeError(
            f"Required dataset '{key}' is not binary."
        )

print("All four required benchmark datasets are available and binary.")

All four required benchmark datasets are available and binary.


## 8. AxelSMOTE Implementation

In [ ]:
class AxelSMOTE(BaseEstimator):
    def __init__(
        self,
        k_neighbors=2,
        n_traits=4,
        theta=0.4,
        alpha=0.4,
        random_state=None
    ):
        self.k_neighbors = k_neighbors
        self.n_traits = n_traits
        self.theta = theta
        self.alpha = alpha
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)

        if X.ndim != 2 or len(X) != len(y):
            raise ValueError("Invalid X/y.")

        if not np.isfinite(X).all():
            raise ValueError("NaN/inf in X.")

        if self.k_neighbors < 1 or self.n_traits < 1:
            raise ValueError("k/t must be >=1.")

        if not 0 <= self.theta <= 1 or not 0 <= self.alpha <= 1:
            raise ValueError("theta/alpha must be in [0,1].")

        self.n_features_in_ = X.shape[1]

        self.traits_ = [
            a.astype(int)
            for a in np.array_split(
                np.arange(X.shape[1]),
                min(self.n_traits, X.shape[1])
            )
            if len(a)
        ]

        labels, counts = np.unique(y, return_counts=True)

        self.class_counts_ = dict(zip(labels, counts))
        self.majority_count_ = int(counts.max())

        self.minority_classes_ = [
            c
            for c, n in self.class_counts_.items()
            if n < self.majority_count_
        ]

        self.rng_ = np.random.default_rng(self.random_state)

        return self

    def _neighbors(self, Xc, idx):
        if len(Xc) <= 1:
            return np.array([], dtype=int)

        k = min(self.k_neighbors, len(Xc) - 1)

        nn = NearestNeighbors(
            n_neighbors=k + 1
        ).fit(Xc)

        ids = nn.kneighbors(
            Xc[idx:idx+1],
            return_distance=False
        )[0]

        return ids[ids != idx][:k]

    @staticmethod
    def _similarity(xb, xn, trait):
        return 1 - np.mean(
            np.abs(xb[trait] - xn[trait])
        )

    def _one(self, Xc):
        bi = int(
            self.rng_.integers(len(Xc))
        )

        xb = Xc[bi]
        xhat = xb.copy()

        ids = self._neighbors(Xc, bi)
        exchanged = []

        if len(ids) == 0:
            return xhat

        for trait in self.traits_:
            selected = self.rng_.choice(
                ids,
                size=len(ids),
                replace=False
            )

            for ni in selected:
                xn = Xc[ni]

                sim = self._similarity(
                    xb,
                    xn,
                    trait
                )

                if (
                    sim > self.theta
                    and self.rng_.random() < self.alpha
                ):
                    lam = float(
                        self.rng_.beta(2, 2)
                    )

                    xhat[trait] = (
                        lam * xb[trait]
                        + (1 - lam) * xn[trait]
                    )

                    exchanged.append(trait)

        if exchanged and self.rng_.random() < self.alpha:
            ranges = np.ptp(Xc, axis=0)

            for trait in exchanged:
                xhat[trait] += (
                    0.05
                    * ranges[trait]
                    * self.rng_.normal(
                        0,
                        1,
                        len(trait)
                    )
                )

        return xhat

    def fit_resample(self, X, y):
        self.fit(X, y)

        X = np.asarray(X, dtype=float)
        y = np.asarray(y)

        Xo = X.copy()
        yo = y.copy()

        for c in self.minority_classes_:
            Xc = X[y == c]

            nnew = self.majority_count_ - len(Xc)

            if len(Xc) <= 1 or nnew <= 0:
                continue

            syn = np.vstack([
                self._one(Xc)
                for _ in range(nnew)
            ])

            Xo = np.vstack([Xo, syn])

            yo = np.concatenate([
                yo,
                np.full(nnew, c, dtype=y.dtype)
            ])

        return Xo, yo

### AxelSMOTE sanity check

In [ ]:
rng = np.random.default_rng(123)

X_demo = np.vstack([
    rng.normal(0, 1, size=(80, 6)),
    rng.normal(2, 1, size=(20, 6)),
])
y_demo = np.array([0] * 80 + [1] * 20)

sampler = AxelSMOTE(
    random_state=123,
    **AXEL_PARAMS
)

X_demo_res, y_demo_res = sampler.fit_resample(
    X_demo,
    y_demo
)

print("Before:", class_distribution(y_demo).to_dict())
print("After: ", class_distribution(y_demo_res).to_dict())
print("Output shape:", X_demo_res.shape)

assert X_demo_res.shape[1] == X_demo.shape[1]
assert len(X_demo_res) == len(y_demo_res)
assert np.isfinite(X_demo_res).all()

Before: {0: 80, 1: 20}
After:  {0: 80, 1: 80}
Output shape: (160, 6)


## 9. Evaluation Functions

In [ ]:
def make_classifier(name, seed):
    if name == "logistic_regression":
        return LogisticRegression(
            max_iter=2000,
            random_state=seed
        )

    if name == "random_forest":
        return RandomForestClassifier(
            n_estimators=300,
            random_state=seed,
            n_jobs=-1
        )

    if name == "svm":
        return SVC(
            probability=True,
            random_state=seed
        )

    raise ValueError(f"Unknown classifier: {name}")


def run_model(
    Xtr,
    ytr,
    Xte,
    yte,
    method,
    seed,
    params=None,
    classifier="logistic_regression",
):
    clf = make_classifier(classifier, seed)

    if method == "class_weight":
        try:
            clf.set_params(class_weight="balanced")
        except ValueError:
            pass

        steps = [
            ("scale", StandardScaler()),
            ("clf", clf),
        ]

    elif method == "smote":
        steps = [
            ("scale", StandardScaler()),
            ("sampler", SMOTE(random_state=seed)),
            ("clf", clf),
        ]

    elif method == "axelsmote":
        steps = [
            ("scale", StandardScaler()),
            (
                "sampler",
                AxelSMOTE(
                    random_state=seed,
                    **(params or {})
                )
            ),
            ("clf", clf),
        ]

    else:
        raise ValueError(f"Unknown method: {method}")

    pipe = Pipeline(steps)

    t = time.perf_counter()
    pipe.fit(Xtr, ytr)
    elapsed = time.perf_counter() - t

    pred = pipe.predict(Xte)
    prob = pipe.predict_proba(Xte)[:, 1]

    return {
        "method": method,
        "seed": seed,
        "accuracy": accuracy_score(yte, pred),
        "balanced_accuracy": balanced_accuracy_score(yte, pred),
        "macro_f1": f1_score(yte, pred, average="macro"),
        "minority_f1": f1_score(
            yte, pred, pos_label=1, zero_division=0
        ),
        "minority_precision": precision_score(
            yte, pred, pos_label=1, zero_division=0
        ),
        "minority_recall": recall_score(
            yte, pred, pos_label=1, zero_division=0
        ),
        "roc_auc": roc_auc_score(yte, prob),
        "pr_auc": average_precision_score(yte, prob),
        "runtime_sec": elapsed,
    }


def prepare_binary_split(X, y, test_size=0.2, split_seed=42):
    Xtr, Xte, ytr_raw, yte_raw = train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=split_seed,
    )

    minority = class_distribution(ytr_raw).idxmin()

    ytr = (ytr_raw == minority).astype(int)
    yte = (yte_raw == minority).astype(int)

    return Xtr, Xte, ytr, yte, minority

## 10. Basic Benchmark

**Controlled/synthetic experiment.** Simple binary imbalance comparing Class Weight, SMOTE, and AxelSMOTE.

In [ ]:
BASIC_PATH = expected_result_file("basic_benchmark_results.csv")
BASIC_EXPECTED = len(CONTROLLED_SEEDS) * len(METHODS)

basic_existing = load_complete_csv(
    BASIC_PATH,
    expected_rows=BASIC_EXPECTED,
    required_columns=["method", "seed", "accuracy", "minority_f1"]
)

if basic_existing is not None:
    basic_results = basic_existing
else:
    rows = []

    for seed in CONTROLLED_SEEDS:
        X, y = make_classification(
            n_samples=2000,
            n_features=20,
            n_informative=10,
            n_redundant=2,
            n_classes=2,
            weights=[0.8, 0.2],
            class_sep=1.0,
            random_state=seed,
        )

        Xtr, Xte, ytr, yte = train_test_split(
            X, y,
            test_size=0.2,
            stratify=y,
            random_state=seed,
        )

        for method in METHODS:
            r = run_model(
                Xtr, ytr, Xte, yte,
                method=method,
                seed=seed,
                params=AXEL_PARAMS if method == "axelsmote" else None,
            )
            r["experiment"] = "basic_benchmark"
            rows.append(r)

        save_checkpoint_rows(rows, BASIC_PATH)

    basic_results = pd.DataFrame(rows)

save_csv(basic_results, BASIC_PATH)
print("Basic benchmark:", basic_results.shape)

Saved 3 rows -> /content/drive/MyDrive/AxelSMOTE/results/basic_benchmark_results.csv
Saved 6 rows -> /content/drive/MyDrive/AxelSMOTE/results/basic_benchmark_results.csv
Saved 9 rows -> /content/drive/MyDrive/AxelSMOTE/results/basic_benchmark_results.csv
Saved 12 rows -> /content/drive/MyDrive/AxelSMOTE/results/basic_benchmark_results.csv
Saved 15 rows -> /content/drive/MyDrive/AxelSMOTE/results/basic_benchmark_results.csv
Saved 18 rows -> /content/drive/MyDrive/AxelSMOTE/results/basic_benchmark_results.csv
Saved 21 rows -> /content/drive/MyDrive/AxelSMOTE/results/basic_benchmark_results.csv
Saved 24 rows -> /content/drive/MyDrive/AxelSMOTE/results/basic_benchmark_results.csv
Saved 27 rows -> /content/drive/MyDrive/AxelSMOTE/results/basic_benchmark_results.csv
Saved 30 rows -> /content/drive/MyDrive/AxelSMOTE/results/basic_benchmark_results.csv
Saved 30 rows -> /content/drive/MyDrive/AxelSMOTE/results/basic_benchmark_results.csv
Basic benchmark: (30, 12)


## 11. Real Dataset Benchmark

Primary experiment: **4 datasets × 3 methods × 10 seeds = 120 rows**. The train/test split is fixed at 80/20 with `random_state=42`; the ten seeds control stochastic model/sampling behavior.

In [ ]:
REAL_PATH = expected_result_file("real_dataset_results.csv")
REAL_EXPECTED = len(REAL_DATASETS) * len(METHODS) * len(SEEDS)

REAL_REQUIRED = [
    "dataset", "method", "seed",
    "accuracy", "balanced_accuracy", "macro_f1",
    "minority_f1", "minority_precision",
    "minority_recall", "roc_auc", "pr_auc",
    "runtime_sec"
]

real_existing = load_complete_csv(
    REAL_PATH,
    expected_rows=REAL_EXPECTED,
    required_columns=REAL_REQUIRED,
)

if real_existing is not None:
    real_results = real_existing
else:
    rows = []

    for key in REAL_DATASETS:
        print(f"\n=== Dataset: {key} ===")

        X, y = load_dataset(key)

        Xtr, Xte, ytr, yte, minority = prepare_binary_split(
            X, y,
            test_size=0.2,
            split_seed=42,
        )

        for method in METHODS:
            for seed in SEEDS:
                r = run_model(
                    Xtr, ytr, Xte, yte,
                    method=method,
                    seed=seed,
                    params=AXEL_PARAMS if method == "axelsmote" else None,
                )

                r["dataset"] = key
                r["minority_class_original"] = int(minority)
                rows.append(r)

                # Immediate checkpoint after each completed run.
                save_checkpoint_rows(rows, REAL_PATH)

    real_results = pd.DataFrame(rows)

save_csv(real_results, REAL_PATH)

assert real_results.shape[0] == 120
print("Real benchmark shape:", real_results.shape)
display(real_results.head())


=== Dataset: wisconsin ===
Saved 1 rows -> /content/drive/MyDrive/AxelSMOTE/results/real_dataset_results.csv
Saved 2 rows -> /content/drive/MyDrive/AxelSMOTE/results/real_dataset_results.csv
Saved 3 rows -> /content/drive/MyDrive/AxelSMOTE/results/real_dataset_results.csv
Saved 4 rows -> /content/drive/MyDrive/AxelSMOTE/results/real_dataset_results.csv
Saved 5 rows -> /content/drive/MyDrive/AxelSMOTE/results/real_dataset_results.csv
Saved 6 rows -> /content/drive/MyDrive/AxelSMOTE/results/real_dataset_results.csv
Saved 7 rows -> /content/drive/MyDrive/AxelSMOTE/results/real_dataset_results.csv
Saved 8 rows -> /content/drive/MyDrive/AxelSMOTE/results/real_dataset_results.csv
Saved 9 rows -> /content/drive/MyDrive/AxelSMOTE/results/real_dataset_results.csv
Saved 10 rows -> /content/drive/MyDrive/AxelSMOTE/results/real_dataset_results.csv
Saved 11 rows -> /content/drive/MyDrive/AxelSMOTE/results/real_dataset_results.csv
Saved 12 rows -> /content/drive/MyDrive/AxelSMOTE/results/real_datas

,method,seed,accuracy,balanced_accuracy,macro_f1,minority_f1,minority_precision,minority_recall,roc_auc,pr_auc,runtime_sec,dataset,minority_class_original
0,class_weight,0,0.957143,0.952446,0.952446,0.9375,0.9375,0.9375,0.995018,0.990538,0.007462,wisconsin,1
1,class_weight,1,0.957143,0.952446,0.952446,0.9375,0.9375,0.9375,0.995018,0.990538,0.008716,wisconsin,1
2,class_weight,2,0.957143,0.952446,0.952446,0.9375,0.9375,0.9375,0.995018,0.990538,0.008576,wisconsin,1
3,class_weight,3,0.957143,0.952446,0.952446,0.9375,0.9375,0.9375,0.995018,0.990538,0.008590,wisconsin,1
4,class_weight,4,0.957143,0.952446,0.952446,0.9375,0.9375,0.9375,0.995018,0.990538,0.009076,wisconsin,1


## 12. Imbalance Experiment

**Controlled/synthetic experiment.** Majority-to-minority ratios are varied over `[2, 5, 10, 20]`.

In [ ]:
IMBALANCE_PATH = expected_result_file("imbalance_results.csv")
IMBALANCE_EXPECTED = (
    len(IMBALANCE_RATIOS)
    * len(METHODS)
    * len(CONTROLLED_SEEDS)
)

imb_existing = load_complete_csv(
    IMBALANCE_PATH,
    expected_rows=IMBALANCE_EXPECTED,
    required_columns=[
        "imbalance_ratio", "method", "seed", "minority_f1"
    ],
)

if imb_existing is not None:
    imbalance_results = imb_existing
else:
    rows = []

    for ratio in IMBALANCE_RATIOS:
        minority_weight = 1.0 / (ratio + 1.0)

        for seed in CONTROLLED_SEEDS:
            X, y = make_classification(
                n_samples=2200,
                n_features=20,
                n_informative=10,
                n_redundant=2,
                n_classes=2,
                weights=[1 - minority_weight, minority_weight],
                class_sep=1.0,
                random_state=seed,
            )

            Xtr, Xte, ytr, yte = train_test_split(
                X, y,
                test_size=0.2,
                stratify=y,
                random_state=seed,
            )

            for method in METHODS:
                r = run_model(
                    Xtr, ytr, Xte, yte,
                    method=method,
                    seed=seed,
                    params=AXEL_PARAMS if method == "axelsmote" else None,
                )
                r.update({
                    "experiment": "imbalance",
                    "imbalance_ratio": ratio,
                })
                rows.append(r)

            save_checkpoint_rows(rows, IMBALANCE_PATH)

    imbalance_results = pd.DataFrame(rows)

save_csv(imbalance_results, IMBALANCE_PATH)
print("Imbalance experiment:", imbalance_results.shape)

Saved 3 rows -> /content/drive/MyDrive/AxelSMOTE/results/imbalance_results.csv
Saved 6 rows -> /content/drive/MyDrive/AxelSMOTE/results/imbalance_results.csv
Saved 9 rows -> /content/drive/MyDrive/AxelSMOTE/results/imbalance_results.csv
Saved 12 rows -> /content/drive/MyDrive/AxelSMOTE/results/imbalance_results.csv
Saved 15 rows -> /content/drive/MyDrive/AxelSMOTE/results/imbalance_results.csv
Saved 18 rows -> /content/drive/MyDrive/AxelSMOTE/results/imbalance_results.csv
Saved 21 rows -> /content/drive/MyDrive/AxelSMOTE/results/imbalance_results.csv
Saved 24 rows -> /content/drive/MyDrive/AxelSMOTE/results/imbalance_results.csv
Saved 27 rows -> /content/drive/MyDrive/AxelSMOTE/results/imbalance_results.csv
Saved 30 rows -> /content/drive/MyDrive/AxelSMOTE/results/imbalance_results.csv
Saved 33 rows -> /content/drive/MyDrive/AxelSMOTE/results/imbalance_results.csv
Saved 36 rows -> /content/drive/MyDrive/AxelSMOTE/results/imbalance_results.csv
Saved 39 rows -> /content/drive/MyDrive/Axe

## 13. Dimensionality Experiment

**Controlled/synthetic experiment.** Number of features: `[5, 10, 20, 50, 100]`.

In [ ]:
DIM_PATH = expected_result_file("dimensionality_results.csv")
DIM_EXPECTED = (
    len(DIMENSIONS)
    * len(METHODS)
    * len(CONTROLLED_SEEDS)
)

dim_existing = load_complete_csv(
    DIM_PATH,
    expected_rows=DIM_EXPECTED,
    required_columns=[
        "n_features", "method", "seed", "minority_f1"
    ],
)

if dim_existing is not None:
    dimensionality_results = dim_existing
else:
    rows = []

    for n_features in DIMENSIONS:
        n_informative = max(
            2,
            min(n_features - 1, n_features // 2)
        )

        for seed in CONTROLLED_SEEDS:
            X, y = make_classification(
                n_samples=2200,
                n_features=n_features,
                n_informative=n_informative,
                n_redundant=0,
                n_classes=2,
                weights=[0.8, 0.2],
                class_sep=1.0,
                random_state=seed,
            )

            Xtr, Xte, ytr, yte = train_test_split(
                X, y,
                test_size=0.2,
                stratify=y,
                random_state=seed,
            )

            for method in METHODS:
                r = run_model(
                    Xtr, ytr, Xte, yte,
                    method=method,
                    seed=seed,
                    params=AXEL_PARAMS if method == "axelsmote" else None,
                )
                r.update({
                    "experiment": "dimensionality",
                    "n_features": n_features,
                })
                rows.append(r)

            save_checkpoint_rows(rows, DIM_PATH)

    dimensionality_results = pd.DataFrame(rows)

save_csv(dimensionality_results, DIM_PATH)
print("Dimensionality experiment:", dimensionality_results.shape)

Saved 3 rows -> /content/drive/MyDrive/AxelSMOTE/results/dimensionality_results.csv
Saved 6 rows -> /content/drive/MyDrive/AxelSMOTE/results/dimensionality_results.csv
Saved 9 rows -> /content/drive/MyDrive/AxelSMOTE/results/dimensionality_results.csv
Saved 12 rows -> /content/drive/MyDrive/AxelSMOTE/results/dimensionality_results.csv
Saved 15 rows -> /content/drive/MyDrive/AxelSMOTE/results/dimensionality_results.csv
Saved 18 rows -> /content/drive/MyDrive/AxelSMOTE/results/dimensionality_results.csv
Saved 21 rows -> /content/drive/MyDrive/AxelSMOTE/results/dimensionality_results.csv
Saved 24 rows -> /content/drive/MyDrive/AxelSMOTE/results/dimensionality_results.csv
Saved 27 rows -> /content/drive/MyDrive/AxelSMOTE/results/dimensionality_results.csv
Saved 30 rows -> /content/drive/MyDrive/AxelSMOTE/results/dimensionality_results.csv
Saved 33 rows -> /content/drive/MyDrive/AxelSMOTE/results/dimensionality_results.csv
Saved 36 rows -> /content/drive/MyDrive/AxelSMOTE/results/dimensiona

## 14. Overlap Experiment

**Controlled/synthetic experiment.** `class_sep = [0.3, 0.6, 1.0, 1.5, 2.0]`. Lower separation represents greater class overlap.

In [ ]:
OVERLAP_PATH = expected_result_file("overlap_results.csv")
OVERLAP_EXPECTED = (
    len(CLASS_SEPARATIONS)
    * len(METHODS)
    * len(CONTROLLED_SEEDS)
)

overlap_existing = load_complete_csv(
    OVERLAP_PATH,
    expected_rows=OVERLAP_EXPECTED,
    required_columns=[
        "class_sep", "method", "seed", "minority_f1"
    ],
)

if overlap_existing is not None:
    overlap_results = overlap_existing
else:
    rows = []

    for class_sep in CLASS_SEPARATIONS:
        for seed in CONTROLLED_SEEDS:
            X, y = make_classification(
                n_samples=2200,
                n_features=20,
                n_informative=10,
                n_redundant=2,
                n_classes=2,
                weights=[0.8, 0.2],
                class_sep=class_sep,
                random_state=seed,
            )

            Xtr, Xte, ytr, yte = train_test_split(
                X, y,
                test_size=0.2,
                stratify=y,
                random_state=seed,
            )

            for method in METHODS:
                r = run_model(
                    Xtr, ytr, Xte, yte,
                    method=method,
                    seed=seed,
                    params=AXEL_PARAMS if method == "axelsmote" else None,
                )
                r.update({
                    "experiment": "overlap",
                    "class_sep": class_sep,
                })
                rows.append(r)

            save_checkpoint_rows(rows, OVERLAP_PATH)

    overlap_results = pd.DataFrame(rows)

save_csv(overlap_results, OVERLAP_PATH)
print("Overlap experiment:", overlap_results.shape)

Saved 3 rows -> /content/drive/MyDrive/AxelSMOTE/results/overlap_results.csv
Saved 6 rows -> /content/drive/MyDrive/AxelSMOTE/results/overlap_results.csv
Saved 9 rows -> /content/drive/MyDrive/AxelSMOTE/results/overlap_results.csv
Saved 12 rows -> /content/drive/MyDrive/AxelSMOTE/results/overlap_results.csv
Saved 15 rows -> /content/drive/MyDrive/AxelSMOTE/results/overlap_results.csv
Saved 18 rows -> /content/drive/MyDrive/AxelSMOTE/results/overlap_results.csv
Saved 21 rows -> /content/drive/MyDrive/AxelSMOTE/results/overlap_results.csv
Saved 24 rows -> /content/drive/MyDrive/AxelSMOTE/results/overlap_results.csv
Saved 27 rows -> /content/drive/MyDrive/AxelSMOTE/results/overlap_results.csv
Saved 30 rows -> /content/drive/MyDrive/AxelSMOTE/results/overlap_results.csv
Saved 33 rows -> /content/drive/MyDrive/AxelSMOTE/results/overlap_results.csv
Saved 36 rows -> /content/drive/MyDrive/AxelSMOTE/results/overlap_results.csv
Saved 39 rows -> /content/drive/MyDrive/AxelSMOTE/results/overlap_r

## 15. AxelSMOTE Sensitivity Analysis

Each parameter is varied one at a time while all other AxelSMOTE parameters remain at their defaults.

In [ ]:
SENS_PATH = expected_result_file("sensitivity_results.csv")

SENS_EXPECTED = sum(
    len(values)
    for values in SENSITIVITY_VALUES.values()
) * len(CONTROLLED_SEEDS)

sens_existing = load_complete_csv(
    SENS_PATH,
    expected_rows=SENS_EXPECTED,
    required_columns=[
        "parameter", "value", "seed", "minority_f1"
    ],
)

if sens_existing is not None:
    sensitivity_results = sens_existing
else:
    rows = []

    for parameter, values in SENSITIVITY_VALUES.items():
        for value in values:
            for seed in CONTROLLED_SEEDS:
                X, y = make_classification(
                    n_samples=2200,
                    n_features=20,
                    n_informative=10,
                    n_redundant=2,
                    n_classes=2,
                    weights=[0.8, 0.2],
                    class_sep=1.0,
                    random_state=seed,
                )

                Xtr, Xte, ytr, yte = train_test_split(
                    X, y,
                    test_size=0.2,
                    stratify=y,
                    random_state=seed,
                )

                params = dict(AXEL_PARAMS)
                params[parameter] = value

                r = run_model(
                    Xtr, ytr, Xte, yte,
                    method="axelsmote",
                    seed=seed,
                    params=params,
                )
                r.update({
                    "experiment": "sensitivity",
                    "parameter": parameter,
                    "value": value,
                })
                rows.append(r)

            save_checkpoint_rows(rows, SENS_PATH)

    sensitivity_results = pd.DataFrame(rows)

save_csv(sensitivity_results, SENS_PATH)
print("Sensitivity experiment:", sensitivity_results.shape)

Saved 10 rows -> /content/drive/MyDrive/AxelSMOTE/results/sensitivity_results.csv
Saved 20 rows -> /content/drive/MyDrive/AxelSMOTE/results/sensitivity_results.csv
Saved 30 rows -> /content/drive/MyDrive/AxelSMOTE/results/sensitivity_results.csv
Saved 40 rows -> /content/drive/MyDrive/AxelSMOTE/results/sensitivity_results.csv
Saved 50 rows -> /content/drive/MyDrive/AxelSMOTE/results/sensitivity_results.csv
Saved 60 rows -> /content/drive/MyDrive/AxelSMOTE/results/sensitivity_results.csv
Saved 70 rows -> /content/drive/MyDrive/AxelSMOTE/results/sensitivity_results.csv
Saved 80 rows -> /content/drive/MyDrive/AxelSMOTE/results/sensitivity_results.csv
Saved 90 rows -> /content/drive/MyDrive/AxelSMOTE/results/sensitivity_results.csv
Saved 100 rows -> /content/drive/MyDrive/AxelSMOTE/results/sensitivity_results.csv
Saved 110 rows -> /content/drive/MyDrive/AxelSMOTE/results/sensitivity_results.csv
Saved 120 rows -> /content/drive/MyDrive/AxelSMOTE/results/sensitivity_results.csv
Saved 130 row

## 16. Classifier Extension

**Controlled/synthetic experiment.** Compare Class Weight, SMOTE, and AxelSMOTE with Logistic Regression, Random Forest, and SVM.

In [ ]:
EXT_PATH = expected_result_file("extension_results.csv")
CLASSIFIERS = [
    "logistic_regression",
    "random_forest",
    "svm",
]

EXT_EXPECTED = (
    len(CLASSIFIERS)
    * len(METHODS)
    * len(CONTROLLED_SEEDS)
)

ext_existing = load_complete_csv(
    EXT_PATH,
    expected_rows=EXT_EXPECTED,
    required_columns=[
        "classifier", "method", "seed", "minority_f1"
    ],
)

if ext_existing is not None:
    extension_results = ext_existing
else:
    rows = []

    for seed in CONTROLLED_SEEDS:
        X, y = make_classification(
            n_samples=2200,
            n_features=20,
            n_informative=10,
            n_redundant=2,
            n_classes=2,
            weights=[0.8, 0.2],
            class_sep=1.0,
            random_state=seed,
        )

        Xtr, Xte, ytr, yte = train_test_split(
            X, y,
            test_size=0.2,
            stratify=y,
            random_state=seed,
        )

        for classifier in CLASSIFIERS:
            for method in METHODS:
                r = run_model(
                    Xtr, ytr, Xte, yte,
                    method=method,
                    seed=seed,
                    params=AXEL_PARAMS if method == "axelsmote" else None,
                    classifier=classifier,
                )
                r.update({
                    "experiment": "extension",
                    "classifier": classifier,
                })
                rows.append(r)

        save_checkpoint_rows(rows, EXT_PATH)

    extension_results = pd.DataFrame(rows)

save_csv(extension_results, EXT_PATH)
print("Classifier extension:", extension_results.shape)

Saved 9 rows -> /content/drive/MyDrive/AxelSMOTE/results/extension_results.csv
Saved 18 rows -> /content/drive/MyDrive/AxelSMOTE/results/extension_results.csv
Saved 27 rows -> /content/drive/MyDrive/AxelSMOTE/results/extension_results.csv
Saved 36 rows -> /content/drive/MyDrive/AxelSMOTE/results/extension_results.csv
Saved 45 rows -> /content/drive/MyDrive/AxelSMOTE/results/extension_results.csv
Saved 54 rows -> /content/drive/MyDrive/AxelSMOTE/results/extension_results.csv
Saved 63 rows -> /content/drive/MyDrive/AxelSMOTE/results/extension_results.csv
Saved 72 rows -> /content/drive/MyDrive/AxelSMOTE/results/extension_results.csv
Saved 81 rows -> /content/drive/MyDrive/AxelSMOTE/results/extension_results.csv
Saved 90 rows -> /content/drive/MyDrive/AxelSMOTE/results/extension_results.csv
Saved 90 rows -> /content/drive/MyDrive/AxelSMOTE/results/extension_results.csv
Classifier extension: (90, 13)


## 17. Statistical Summary

In [ ]:
METRICS = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "minority_f1",
    "minority_precision",
    "minority_recall",
    "roc_auc",
    "pr_auc",
    "runtime_sec",
]


def mean_std_summary(df, group_cols):
    rows = []

    for keys, g in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = dict(zip(group_cols, keys))

        for metric in METRICS:
            row[f"{metric}_mean"] = g[metric].mean()
            row[f"{metric}_std"] = g[metric].std(ddof=1)

        rows.append(row)

    return pd.DataFrame(rows)


REAL_SUMMARY_PATH = TABLES_DIR / "real_dataset_summary.csv"
OVERALL_SUMMARY_PATH = TABLES_DIR / "overall_method_summary.csv"

real_summary = mean_std_summary(
    real_results,
    ["dataset", "method"],
)

overall_summary = mean_std_summary(
    real_results,
    ["method"],
)

save_csv(real_summary, REAL_SUMMARY_PATH)
save_csv(overall_summary, OVERALL_SUMMARY_PATH)

display(real_summary)
display(overall_summary)

Saved 12 rows -> /content/drive/MyDrive/AxelSMOTE/final_tables/real_dataset_summary.csv
Saved 3 rows -> /content/drive/MyDrive/AxelSMOTE/final_tables/overall_method_summary.csv


,dataset,method,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std,minority_f1_mean,minority_f1_std,minority_precision_mean,minority_precision_std,minority_recall_mean,minority_recall_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,runtime_sec_mean,runtime_sec_std
0,ads,axelsmote,0.966921,2.161793e-03,0.936189,3.591901e-03,0.932074,4.185518e-03,0.883422,7.103623e-03,0.873682,1.192702e-02,0.893478,6.874517e-03,0.979575,1.350001e-03,0.941684,2.917994e-03,15.029094,1.283747
1,ads,class_weight,0.966463,1.170278e-16,0.935014,0.000000e+00,0.931091,2.340556e-16,0.881720,1.170278e-16,0.872340,0.000000e+00,0.891304,2.340556e-16,0.981961,2.340556e-16,0.948001,1.170278e-16,2.899540,0.804491
2,ads,smote,0.965244,2.945402e-03,0.940672,3.364204e-03,0.929748,5.475165e-03,0.879812,9.204217e-03,0.854764,1.593841e-02,0.906522,5.613019e-03,0.977303,1.081160e-03,0.940719,2.739638e-03,4.270213,1.023544
3,ilpd,axelsmote,0.641880,1.171214e-02,0.705918,9.726339e-03,0.634306,1.030960e-02,0.582236,9.146987e-03,0.440608,8.653795e-03,0.858824,2.702755e-02,0.740751,9.561410e-03,0.521686,1.261968e-02,0.465358,0.071604
4,ilpd,class_weight,0.632479,1.170278e-16,0.697555,0.000000e+00,0.625475,0.000000e+00,0.574257,1.170278e-16,0.432836,5.851389e-17,0.852941,1.170278e-16,0.733168,1.170278e-16,0.516216,1.170278e-16,0.015323,0.013399
5,ilpd,smote,0.650427,6.306537e-03,0.715415,5.613972e-03,0.642875,4.960609e-03,0.591348,5.201302e-03,0.447891,4.263479e-03,0.870588,2.480218e-02,0.745500,1.213105e-02,0.528038,1.466642e-02,0.015016,0.001463
6,kc1,axelsmote,0.745972,7.392960e-03,0.707037,6.268602e-03,0.638397,7.011401e-03,0.441181,8.743423e-03,0.333760,8.869723e-03,0.650769,7.431475e-03,0.799444,4.107730e-03,0.469766,9.276811e-03,1.441162,0.177082
7,kc1,class_weight,0.739336,0.000000e+00,0.701228,1.170278e-16,0.631879,1.170278e-16,0.432990,1.170278e-16,0.325581,5.851389e-17,0.646154,1.170278e-16,0.802629,0.000000e+00,0.477920,1.170278e-16,0.027061,0.004833
8,kc1,smote,0.731754,6.958204e-03,0.694859,1.048912e-02,0.624651,7.204065e-03,0.424180,1.100450e-02,0.316926,8.363631e-03,0.641538,2.299129e-02,0.794954,3.101346e-03,0.464395,1.321449e-02,0.042349,0.013499
9,wisconsin,axelsmote,0.957143,1.170278e-16,0.952446,2.340556e-16,0.952446,2.340556e-16,0.937500,0.000000e+00,0.937500,0.000000e+00,0.937500,0.000000e+00,0.994792,2.386985e-04,0.990054,5.432359e-04,0.284999,0.018947


,method,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std,minority_f1_mean,minority_f1_std,minority_precision_mean,minority_precision_std,minority_recall_mean,minority_recall_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,runtime_sec_mean,runtime_sec_std
0,axelsmote,0.827979,0.140988,0.825398,0.120717,0.789306,0.155210,0.711085,0.209152,0.646387,0.266389,0.835143,0.112300,0.878641,0.112162,0.730798,0.239541,4.305153,6.316998
1,class_weight,0.823855,0.144888,0.821561,0.123890,0.785223,0.158741,0.706617,0.212651,0.642064,0.269970,0.831975,0.112802,0.878194,0.114532,0.733169,0.239985,0.737697,1.321834
2,smote,0.826142,0.139947,0.825848,0.122687,0.787430,0.156052,0.708210,0.212772,0.639270,0.266130,0.839037,0.119086,0.878171,0.111088,0.730878,0.239593,1.085031,1.926253


### Compact research table

In [ ]:
compact = real_summary[
    [
        "dataset", "method",
        "balanced_accuracy_mean", "balanced_accuracy_std",
        "minority_f1_mean", "minority_f1_std",
        "roc_auc_mean", "roc_auc_std",
        "pr_auc_mean", "pr_auc_std",
        "runtime_sec_mean", "runtime_sec_std",
    ]
].sort_values(["dataset", "method"])

display(compact)

,dataset,method,balanced_accuracy_mean,balanced_accuracy_std,minority_f1_mean,minority_f1_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,runtime_sec_mean,runtime_sec_std
0,ads,axelsmote,0.936189,3.591901e-03,0.883422,7.103623e-03,0.979575,1.350001e-03,0.941684,2.917994e-03,15.029094,1.283747
1,ads,class_weight,0.935014,0.000000e+00,0.881720,1.170278e-16,0.981961,2.340556e-16,0.948001,1.170278e-16,2.899540,0.804491
2,ads,smote,0.940672,3.364204e-03,0.879812,9.204217e-03,0.977303,1.081160e-03,0.940719,2.739638e-03,4.270213,1.023544
3,ilpd,axelsmote,0.705918,9.726339e-03,0.582236,9.146987e-03,0.740751,9.561410e-03,0.521686,1.261968e-02,0.465358,0.071604
4,ilpd,class_weight,0.697555,0.000000e+00,0.574257,1.170278e-16,0.733168,1.170278e-16,0.516216,1.170278e-16,0.015323,0.013399
5,ilpd,smote,0.715415,5.613972e-03,0.591348,5.201302e-03,0.745500,1.213105e-02,0.528038,1.466642e-02,0.015016,0.001463
6,kc1,axelsmote,0.707037,6.268602e-03,0.441181,8.743423e-03,0.799444,4.107730e-03,0.469766,9.276811e-03,1.441162,0.177082
7,kc1,class_weight,0.701228,1.170278e-16,0.432990,1.170278e-16,0.802629,0.000000e+00,0.477920,1.170278e-16,0.027061,0.004833
8,kc1,smote,0.694859,1.048912e-02,0.424180,1.100450e-02,0.794954,3.101346e-03,0.464395,1.321449e-02,0.042349,0.013499
9,wisconsin,axelsmote,0.952446,2.340556e-16,0.937500,0.000000e+00,0.994792,2.386985e-04,0.990054,5.432359e-04,0.284999,0.018947


In [ ]:
print("Real benchmark:")
display(real_results.groupby("method")[[
    "minority_f1",
    "pr_auc",
    "runtime_sec"
]].mean().round(4))

print("\nImbalance experiment:")
display(
    imbalance_results
    .groupby(["imbalance_ratio", "method"])["minority_f1"]
    .mean()
    .round(4)
    .reset_index()
)

print("\nDimensionality experiment:")
display(
    dimensionality_results
    .groupby(["n_features", "method"])["minority_f1"]
    .mean()
    .round(4)
    .reset_index()
)

print("\nOverlap experiment:")
display(
    overlap_results
    .groupby(["class_sep", "method"])["minority_f1"]
    .mean()
    .round(4)
    .reset_index()
)

print("\nSensitivity experiment:")
display(
    sensitivity_results
    .groupby(["parameter", "value"])["minority_f1"]
    .mean()
    .round(4)
    .reset_index()
)

print("\nClassifier extension:")
display(
    extension_results
    .groupby(["classifier", "method"])["minority_f1"]
    .mean()
    .round(4)
    .reset_index()
)

Real benchmark:


,minority_f1,pr_auc,runtime_sec
method,,,
axelsmote,0.7111,0.7308,4.3052
class_weight,0.7066,0.7332,0.7377
smote,0.7082,0.7309,1.0850



Imbalance experiment:


,imbalance_ratio,method,minority_f1
0,2,axelsmote,0.7888
1,2,class_weight,0.7877
2,2,smote,0.7926
3,5,axelsmote,0.6476
4,5,class_weight,0.6494
5,5,smote,0.6550
6,10,axelsmote,0.4980
7,10,class_weight,0.5023
8,10,smote,0.5175
9,20,axelsmote,0.3044



Dimensionality experiment:


,n_features,method,minority_f1
0,5,axelsmote,0.7939
1,5,class_weight,0.7922
2,5,smote,0.7959
3,10,axelsmote,0.7283
4,10,class_weight,0.7338
5,10,smote,0.7323
6,20,axelsmote,0.6925
7,20,class_weight,0.6913
8,20,smote,0.6989
9,50,axelsmote,0.6440


... (2 outputs left out)


Sensitivity experiment:


,parameter,value,minority_f1
0,alpha,0.2,0.6968
1,alpha,0.4,0.6982
2,alpha,0.6,0.6962
3,alpha,0.8,0.6973
4,k_neighbors,1.0,0.6992
5,k_neighbors,2.0,0.6982
6,k_neighbors,3.0,0.6958
7,k_neighbors,5.0,0.6982
8,n_traits,2.0,0.6949
9,n_traits,4.0,0.6982



Classifier extension:


,classifier,method,minority_f1
0,logistic_regression,axelsmote,0.6982
1,logistic_regression,class_weight,0.7001
2,logistic_regression,smote,0.6991
3,random_forest,axelsmote,0.8450
4,random_forest,class_weight,0.7533
5,random_forest,smote,0.8441
6,svm,axelsmote,0.8674
7,svm,class_weight,0.8555
8,svm,smote,0.8553


## 19. Final Output Verification

Creates a manifest containing the result files, sizes, row counts, and completeness status.

In [ ]:
EXPECTED_FILES = {
    "basic_benchmark_results.csv": BASIC_EXPECTED,
    "imbalance_results.csv": IMBALANCE_EXPECTED,
    "dimensionality_results.csv": DIM_EXPECTED,
    "overlap_results.csv": OVERLAP_EXPECTED,
    "sensitivity_results.csv": SENS_EXPECTED,
    "real_dataset_results.csv": REAL_EXPECTED,
    "extension_results.csv": EXT_EXPECTED,
}

manifest_rows = []

for filename, expected_rows in EXPECTED_FILES.items():
    path = RESULTS_DIR / filename

    if path.exists():
        try:
            df = pd.read_csv(path)
            manifest_rows.append({
                "file": filename,
                "exists": True,
                "size_bytes": path.stat().st_size,
                "rows": len(df),
                "expected_rows": expected_rows,
                "complete": len(df) == expected_rows,
            })
        except Exception as exc:
            manifest_rows.append({
                "file": filename,
                "exists": True,
                "size_bytes": path.stat().st_size,
                "rows": np.nan,
                "expected_rows": expected_rows,
                "complete": False,
                "error": str(exc),
            })
    else:
        manifest_rows.append({
            "file": filename,
            "exists": False,
            "size_bytes": 0,
            "rows": 0,
            "expected_rows": expected_rows,
            "complete": False,
        })

for filename in [
    "real_dataset_summary.csv",
    "overall_method_summary.csv",
]:
    path = TABLES_DIR / filename

    if path.exists():
        df = pd.read_csv(path)
        manifest_rows.append({
            "file": f"final_tables/{filename}",
            "exists": True,
            "size_bytes": path.stat().st_size,
            "rows": len(df),
            "expected_rows": np.nan,
            "complete": True,
        })
    else:
        manifest_rows.append({
            "file": f"final_tables/{filename}",
            "exists": False,
            "size_bytes": 0,
            "rows": 0,
            "expected_rows": np.nan,
            "complete": False,
        })

manifest_df = pd.DataFrame(manifest_rows)

MANIFEST_PATH = RESULTS_DIR / "manifest.csv"
save_csv(manifest_df, MANIFEST_PATH)

display(manifest_df)

print("\nProject directories:")
for directory in [
    DRIVE_ROOT,
    RESULTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
]:
    print(f"\n{directory}")
    if directory.exists():
        for p in sorted(directory.iterdir()):
            print("  ", p.name)

assert manifest_df.loc[
    manifest_df["file"] == "real_dataset_results.csv",
    "complete"
].all()

print("\nFinal verification complete.")

Saved 9 rows -> /content/drive/MyDrive/AxelSMOTE/results/manifest.csv


,file,exists,size_bytes,rows,expected_rows,complete
0,basic_benchmark_results.csv,True,5540,30,30.0,True
1,imbalance_results.csv,True,23092,120,120.0,True
2,dimensionality_results.csv,True,29664,150,150.0,True
3,overlap_results.csv,True,28943,150,150.0,True
4,sensitivity_results.csv,True,32562,160,160.0,True
5,real_dataset_results.csv,True,21760,120,120.0,True
6,extension_results.csv,True,18286,90,90.0,True
7,final_tables/real_dataset_summary.csv,True,4476,12,NaN,True
8,final_tables/overall_method_summary.csv,True,1373,3,NaN,True



Project directories:

/content/drive/MyDrive/AxelSMOTE
   figures
   final_tables
   results

/content/drive/MyDrive/AxelSMOTE/results
   basic_benchmark_results.csv
   dimensionality_results.csv
   extension_results.csv
   imbalance_results.csv
   manifest.csv
   overlap_results.csv
   real_dataset_results.csv
   sensitivity_results.csv

/content/drive/MyDrive/AxelSMOTE/figures

/content/drive/MyDrive/AxelSMOTE/final_tables
   overall_method_summary.csv
   real_dataset_summary.csv

Final verification complete.


In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/AxelSMOTE")

files = [
    "final_tables/overall_method_summary.csv",
    "final_tables/real_dataset_summary.csv",
    "results/imbalance_results.csv",
    "results/dimensionality_results.csv",
    "results/overlap_results.csv",
    "results/sensitivity_results.csv",
    "results/extension_results.csv",
]

for file in files:
    path = ROOT / file
    print("\n" + "="*80)
    print(file)
    print("="*80)
    df = pd.read_csv(path)
    print(df.to_string(index=False))


final_tables/overall_method_summary.csv
      method  accuracy_mean  accuracy_std  balanced_accuracy_mean  balanced_accuracy_std  macro_f1_mean  macro_f1_std  minority_f1_mean  minority_f1_std  minority_precision_mean  minority_precision_std  minority_recall_mean  minority_recall_std  roc_auc_mean  roc_auc_std  pr_auc_mean  pr_auc_std  runtime_sec_mean  runtime_sec_std
   axelsmote       0.827979      0.140988                0.825398               0.120717       0.789306      0.155210          0.711085         0.209152                 0.646387                0.266389              0.835143             0.112300      0.878641     0.112162     0.730798    0.239541          4.305153         6.316998
class_weight       0.823855      0.144888                0.821561               0.123890       0.785223      0.158741          0.706617         0.212651                 0.642064                0.269970              0.831975             0.112802      0.878194     0.114532     0.733169    0.23998

## Research Interpretation Reminder

The notebook reports results; it does not assume AxelSMOTE is superior.

When interpreting results:
1. Use all ten seeds.
2. Report mean and standard deviation.
3. Examine minority F1, minority recall, PR-AUC, and balanced accuracy rather than accuracy alone.
4. Consider runtime alongside predictive performance.
5. Treat synthetic experiments as controlled diagnostics, not proof of real-world superiority.
6. Preserve unfavorable results.
7. Before describing this as a paper reproduction, verify the custom AxelSMOTE implementation against the original paper's exact algorithm and parameter definitions.